# Classificação de Peças Processuais — STF (VICTOR)
Pipeline: TF-IDF sobre `text_clean` + 44 features estruturais pré-extraídas, treinado com LightGBM, validado por F1-Score Macro em StratifiedKFold, com pesos de classe para lidar com o forte desbalanceamento (classe 0 e 2 são minoritárias).

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

import lightgbm as lgb

RANDOM_STATE = 42

## Carregamento dos dados

In [ ]:
train = pd.read_csv('/kaggle/input/train.csv')
test = pd.read_csv('/kaggle/input/test.csv')
sample_submission = pd.read_csv('/kaggle/input/sample_submission.csv')

print(train.shape, test.shape)
train.head()

## Distribuição das classes
Dataset fortemente desbalanceado — RE (classe 3) domina, enquanto Acórdão (0) e Despacho (2) são raras.

In [ ]:
class_names = {0: 'Acórdão', 1: 'ARE', 2: 'Despacho', 3: 'RE', 4: 'Sentença'}

counts = train['Category'].value_counts().sort_index()
counts.index = counts.index.map(class_names)

plt.figure(figsize=(7,4))
counts.plot(kind='bar', color='#4C72B0')
plt.ylabel('Quantidade de amostras')
plt.title('Distribuição das classes no treino')
plt.tight_layout()
plt.show()

counts

## Tratamento de valores nulos no texto
Algumas linhas de `text_clean` estão vazias (NaN); preenchemos com string vazia para não quebrar o vetorizador.

In [ ]:
train['text_clean'] = train['text_clean'].fillna('')
test['text_clean'] = test['text_clean'].fillna('')

## Separação de features numéricas e alvo
As 44 colunas de contagem/keywords entram como features estruturais; `text_clean` alimenta o TF-IDF.

In [ ]:
id_cols = ['Id', 'Category', 'Body', 'text_clean', 'text_head_30']
numeric_cols = [c for c in train.columns if c not in id_cols]
print(len(numeric_cols), 'features numéricas')

y = train['Category'].values
X_train_text = train['text_clean']
X_test_text = test['text_clean']
X_train_num = train[numeric_cols].fillna(0)
X_test_num = test[numeric_cols].fillna(0)

## Vetorização TF-IDF
Unigramas e bigramas, ajustados apenas no treino para evitar vazamento de informação do teste.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

X_train_tfidf.shape, X_test_tfidf.shape

## Combinação texto + features numéricas
As features numéricas são padronizadas e concatenadas horizontalmente com a matriz esparsa de TF-IDF.

In [ ]:
scaler = StandardScaler()
X_train_num_scaled = sp.csr_matrix(scaler.fit_transform(X_train_num))
X_test_num_scaled = sp.csr_matrix(scaler.transform(X_test_num))

X_train_full = sp.hstack([X_train_tfidf, X_train_num_scaled]).tocsr()
X_test_full = sp.hstack([X_test_tfidf, X_test_num_scaled]).tocsr()

X_train_full.shape, X_test_full.shape

## Pesos de classe
Compensam o desbalanceamento durante o treino, dando mais peso às classes raras (Acórdão, Despacho).

In [ ]:
sample_weights = compute_sample_weight(class_weight='balanced', y=y)

## Validação cruzada (StratifiedKFold, F1 Macro)
Avalia a capacidade de generalização do modelo antes do treino final, mantendo a proporção de classes em cada fold.

In [ ]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_preds = np.zeros(len(y), dtype=int)

lgb_params = dict(
    objective='multiclass',
    num_class=5,
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y)):
    X_tr, X_val = X_train_full[train_idx], X_train_full[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    w_tr = sample_weights[train_idx]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr, y_tr,
        sample_weight=w_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )

    val_pred = model.predict(X_val)
    oof_preds[val_idx] = val_pred

    fold_f1 = f1_score(y_val, val_pred, average='macro')
    print(f'Fold {fold+1}: F1 Macro = {fold_f1:.4f}')

overall_f1 = f1_score(y, oof_preds, average='macro')
print(f'\nF1 Macro (out-of-fold): {overall_f1:.4f}')

## Relatório de classificação (out-of-fold)
Detalha precisão, recall e F1 por classe — útil para identificar onde o modelo mais erra.

In [ ]:
print(classification_report(y, oof_preds, target_names=[class_names[i] for i in range(5)]))

cm = confusion_matrix(y, oof_preds)
plt.figure(figsize=(6,5))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks(range(5), [class_names[i] for i in range(5)], rotation=45)
plt.yticks(range(5), [class_names[i] for i in range(5)])
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de confusão (out-of-fold)')
for i in range(5):
    for j in range(5):
        plt.text(j, i, cm[i, j], ha='center', va='center')
plt.tight_layout()
plt.show()

## Treino final e predição no teste
Modelo treinado com todo o conjunto de treino, usando o número médio de árvores obtido na validação cruzada.

In [ ]:
final_model = lgb.LGBMClassifier(**lgb_params)
final_model.fit(X_train_full, y, sample_weight=sample_weights)

test_preds = final_model.predict(X_test_full)

## Geração do arquivo de submissão

In [ ]:
submission = pd.DataFrame({
    'Id': test['Id'],
    'Category': test_preds
})

submission.to_csv('submission.csv', index=False)
submission.head()